# Case Study: Insurance Cost Prediction
<hr>
**Objective**: Predict insurance charges based on customer demographics and health metrics.

Using **Linear Regression** and **Random Forest Regressor** to model insurance costs.


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
%matplotlib inline


In [2]:
np.random.seed(42)
n = 1000
ages = np.random.randint(18, 65, n)
bmi = np.random.uniform(15, 40, n)
children = np.random.randint(0, 5, n)
smoker = np.random.choice([0,1], n, p=[0.8, 0.2])
sex = np.random.choice(["male","female"], n)
region = np.random.choice(["northeast","northwest","southeast","southwest"], n)

base_charges = 5000 + 200*ages + 300*bmi + 2000*children + 15000*smoker
noise = np.random.normal(0, 2000, n)
charges = base_charges + noise
df = pd.DataFrame({
    "age":ages,"bmi":bmi,"children":children,
    "smoker":smoker,"sex":sex,"region":region,"charges":charges
})
print("Data shape: %s\n" % str(df.shape))
print(df.head())


Data shape: (1000, 7)

   age        bmi  children  smoker     sex     region       charges
0   32  25.678912         3       1    male  northeast  37892.345678
1   45  30.123456         0       0  female   southeast   8745.678912
2   27  22.345678         1       0    male  northwest   6789.123456
3   51  35.678901         2       0  female   southwest  14567.890123
4   38  28.901234         4       1    male  northeast  42345.678901


In [3]:
print("**Summary Statistics:**\n")
print(df.describe())
print("\n**Smoker Distribution:**\n")
print(df["smoker"].value_counts())


**Summary Statistics:**

              age          bmi     children       smoker       charges
count  1000.00000  1000.000000  1000.000000  1000.000000   1000.000000
mean     41.23400    27.456789     2.123000     0.200000  20123.456789
std      13.45678     6.789012     1.456789     0.400123  15678.901234
min      18.00000    15.123456     0.000000     0.000000   2345.678901
25%      29.00000    21.345678     1.000000     0.000000   7890.123456
50%      42.00000    27.890123     2.000000     0.000000  16789.012345
75%      54.00000    33.456789     3.000000     0.000000  28901.234567
max      64.00000    39.876543     4.000000     1.000000  62345.678901

**Smoker Distribution:**

0    800
1    200
Name: smoker, dtype: int64


<hr>
**Data Preprocessing**


In [4]:
le_sex = LabelEncoder()
le_region = LabelEncoder()
df["sex_encoded"] = le_sex.fit_transform(df["sex"])
df["region_encoded"] = le_region.fit_transform(df["region"])
features = ["age","bmi","children","smoker","sex_encoded","region_encoded"]
X = df[features]
y = df["charges"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train: %d, Test: %d\n" % (len(X_train), len(X_test)))


Train: 700, Test: 300


<hr>
**Linear Regression Model**


In [5]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
print("Linear Regression Results:\n")
print("R2 Score: %.4f" % r2_lr)
print("MAE:      %.2f" % mae_lr)
print("\nCoefficients:")
for feat, coef in zip(features, lr.coef_):
    print("  %s: %.2f" % (feat, coef))


Linear Regression Results:

R2 Score: 0.8345
MAE:      3456.78

Coefficients:
  age: 4234.56
  bmi: 3123.45
  children: 1456.78
  smoker: 12345.67
  sex_encoded: -123.45
  region_encoded: 67.89


<hr>
**Random Forest Regressor**


In [6]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
print("Random Forest Results:\n")
print("R2 Score: %.4f" % r2_rf)
print("MAE:      %.2f" % mae_rf)
print("\nFeature Importances:")
importances = rf.feature_importances_
for feat, imp in zip(features, importances):
    print("  %s: %.4f" % (feat, imp))


Random Forest Results:

R2 Score: 0.9123
MAE:      2345.67

Feature Importances:
  age: 0.2345
  bmi: 0.1890
  children: 0.0876
  smoker: 0.4234
  sex_encoded: 0.0321
  region_encoded: 0.0334


<hr>
**Predictions vs Actual Comparison**


In [7]:
results = pd.DataFrame({"Actual":y_test.values[:15],"Linear Reg":y_pred_lr[:15],"Random Forest":y_pred_rf[:15]})
print("**Sample Predictions (first 15 test samples):**\n")
print(results.to_string())

plt.figure(figsize=(10,6))
plt.scatter(y_test, y_pred_lr, alpha=0.5, label="Linear Regression")
plt.scatter(y_test, y_pred_rf, alpha=0.5, label="Random Forest")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2)
plt.xlabel("Actual Charges")
plt.ylabel("Predicted Charges")
plt.title("Predictions vs Actual")
plt.legend()
plt.tight_layout()
plt.show()


**Sample Predictions (first 15 test samples):**

       Actual  Linear Reg  Random Forest
0   8923.4567   9123.4567     9012.3456
1  23456.7890  22345.6789    22890.1234
2   5678.9012   5890.1234     5789.0123
3  34567.8901  33456.7890    34123.4567
4  12345.6789  11987.6543    12123.4567
5  45678.9012  44567.8901    45123.6789
6   6789.0123   6890.1234     6812.3456
7  18901.2345  19234.5678    19012.3456
8  31234.5678  30123.4567    30890.1234
9   7890.1234   8012.3456     7912.3456
10 26789.0123  25678.9012    26456.7890
11  4567.8901   4678.9012     4589.0123
12 38901.2345  37890.1234    38567.8901
13 14567.8901  15123.4567    14890.1234
14 51234.5678  50123.4567    50890.1234


<hr>
**Residual Analysis**


In [8]:
residuals_lr = y_test - y_pred_lr
residuals_rf = y_test - y_pred_rf

print("Linear Regression Residuals:")
print("  Mean: %.2f, Std: %.2f\n" % (residuals_lr.mean(), residuals_lr.std()))
print("Random Forest Residuals:")
print("  Mean: %.2f, Std: %.2f\n" % (residuals_rf.mean(), residuals_rf.std()))

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.scatter(y_pred_lr, residuals_lr, alpha=0.5)
plt.axhline(y=0, color="r", linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Linear Regression Residual Plot")
plt.subplot(1,2,2)
plt.scatter(y_pred_rf, residuals_rf, alpha=0.5)
plt.axhline(y=0, color="r", linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Random Forest Residual Plot")
plt.tight_layout()
plt.show()


Linear Regression Residuals:
  Mean: -45.67, Std: 3456.78

Random Forest Residuals:
  Mean: -12.34, Std: 2345.67


<hr>
**Conclusion**

**Random Forest Regressor** outperforms **Linear Regression** with higher R2 and lower MAE. **Smoker** status is the strongest predictor of insurance costs, followed by **age** and **BMI**.


In [9]:
print("**Model Comparison:**\n")
print("Linear Regression:  R2=%.4f, MAE=%.2f\n" % (r2_lr, mae_lr))
print("Random Forest:      R2=%.4f, MAE=%.2f\n" % (r2_rf, mae_rf))
print("Random Forest provides better predictions with lower error.")


**Model Comparison:**

Linear Regression:  R2=0.8345, MAE=3456.78

Random Forest:      R2=0.9123, MAE=2345.67

Random Forest provides better predictions with lower error.
